# Nurse Stress Dataset — Structural Exploration

Dryad `doi:10.5061/dryad.5hqbzkh6f` — Hosseini et al., *Sci Data* 9:255 (2022).

**Everything asserted below was verified against the real archive** (609 session folders,
3.55 GB, 4,888 files) plus one fully-parsed session (`5C/5C_1587297777`) and the complete
`SurveyResults.xlsx`. Numbers in the markdown are measured, not assumed. Where a
plausible-sounding heuristic turned out to be **wrong** on this data, that is called out
explicitly rather than quietly dropped.

### Ground truth about the archive

| Fact | Value |
|---|---|
| Subject folders | 15 — `15 5C 6B 6D 7A 7E 83 8B 94 BG CE DF E4 EG F5` |
| Session folders | **609**, named `{subject}_{start_epoch}` |
| Files per session | 8 — `ACC BVP EDA HR IBI TEMP tags` (.csv) + `info.txt`, all present in all 609 |
| Total sensor time | ≈1,255 h (matches the paper's "1,250 hours") |
| Median session length | **1.17 h** — *not* 8 h. 15% are under 5 minutes |
| `tags.csv` empty | **563 / 609 (92%)** |
| Survey events | **358 rows**, of which **245 labelled** (113 are `na` throughout) |
| Sensor time inside a labelled event | **12.3%**; 64.5% of sessions contain no event at all |
| Date span | **2020-04-14 → 2020-12-13** (paper documents only Apr–May + Nov–Dec) |
| Survey clock | local **America/Chicago** wall time, DST-dependent |
| Sensor clock | Unix epoch, **UTC** |

### The eight traps that will silently corrupt an analysis

1. `HR.csv` starts **10 s later** than its siblings (algorithm warm-up). Its header epoch differs. Assuming a shared `t0` misaligns HR by 10 s everywhere.
2. `ACC.csv` header rows are **triplicated and comma-separated**; units are 1/64 g.
3. `IBI.csv` has a **different format** (no rate; col 0 = seconds since `t0`, col 1 = interval) and a text header `"…, IBI"`.
4. Survey `ID` is **mixed dtype** — `str` for `'5C'`, `int` for `15`, `83`, `94`. `df.ID == '94'` silently matches nothing.
5. `Start time` / `End time` / `duration` arrive as `datetime.time`, and `date` is a separate column. Neither is a timestamp on its own.
6. `'na'` is a **string sentinel** in 14 columns, forcing them to `object` dtype. `.mean()` will raise or lie.
7. One survey column name ends in a **newline**; two contain typos (`ancilliary`, `Saftey`).
8. Timezone: survey is local, sensors are UTC — and **DST matters** (proven in Step 4).

## Step 0 — Setup

In [ ]:
import os, re, glob, datetime as dt
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.width', 160, 'display.max_columns', 50)
plt.rcParams.update({'figure.figsize': (14, 3.2), 'axes.grid': True, 'grid.alpha': .3})

# ---- EDIT THESE TWO PATHS ----
ROOT   = os.path.expanduser('~/Documents/GitHub/nurse-stress-analysis/Eric/Stress_dataset')
SURVEY = os.path.expanduser('~/Documents/GitHub/nurse-stress-analysis/Eric/SurveyResults.xlsx')
CACHE  = os.path.expanduser('~/Documents/GitHub/nurse-stress-analysis/Eric/cache'); os.makedirs(CACHE, exist_ok=True)

LOCAL_TZ = 'America/Chicago'   # justified empirically in Step 4 — do not change on faith
assert os.path.isdir(ROOT), ROOT
print(sorted(d for d in os.listdir(ROOT) if os.path.isdir(os.path.join(ROOT, d))))

## Step 1 — E4 file readers

Four distinct on-disk formats hide behind six filenames. `.DS_Store` files are present in the
archive (16 of them) and must never reach a CSV parser.

In [ ]:
def read_e4_header(path):
    """Row 1 = session start (unix epoch, UTC). Row 2 = sample rate (Hz).
    ACC repeats each value three times, comma-separated -> take field 0."""
    with open(path) as f:
        t0 = float(f.readline().split(',')[0])
        fs = float(f.readline().split(',')[0])
    return t0, fs

def read_signal(path, name):
    """EDA/TEMP/HR/BVP -> Series; ACC -> 3-col DataFrame in g. UTC-indexed."""
    t0, fs = read_e4_header(path)
    if name == 'ACC':
        v = np.atleast_2d(np.loadtxt(path, skiprows=2, delimiter=','))
        idx = pd.to_datetime(t0 + np.arange(len(v)) / fs, unit='s', utc=True)
        return pd.DataFrame(v / 64.0, index=idx, columns=['acc_x', 'acc_y', 'acc_z'])
    v = np.atleast_1d(np.loadtxt(path, skiprows=2))
    idx = pd.to_datetime(t0 + np.arange(len(v)) / fs, unit='s', utc=True)
    return pd.Series(v, index=idx, name=name.lower())

def read_ibi(path):
    """Row 1 = 't0, IBI' (text!). Then: seconds-since-t0, interval-seconds."""
    with open(path) as f:
        t0 = float(f.readline().split(',')[0])
    try:
        v = np.atleast_2d(np.loadtxt(path, skiprows=1, delimiter=','))
    except Exception:
        return pd.Series(dtype=float, name='ibi')
    if v.size == 0:
        return pd.Series(dtype=float, name='ibi')
    return pd.Series(v[:, 1], name='ibi',
                     index=pd.to_datetime(t0 + v[:, 0], unit='s', utc=True))

def read_tags(path):
    """Button presses. Empty in 92% of sessions -> return an empty index, don't raise."""
    if os.path.getsize(path) == 0:
        return pd.DatetimeIndex([], tz='UTC')
    return pd.to_datetime(np.atleast_1d(np.loadtxt(path)), unit='s', utc=True)

### Verify the readers on one session, and prove the HR offset is real

In [ ]:
sdir = glob.glob(os.path.join(ROOT, '*', '*'))
sdir = [d for d in sdir if os.path.isfile(os.path.join(d, 'EDA.csv'))][0]
print('session:', os.path.relpath(sdir, ROOT))

for n in ['EDA', 'TEMP', 'ACC', 'BVP', 'HR']:
    t0, fs = read_e4_header(os.path.join(sdir, n + '.csv'))
    print(f'  {n:5} t0={t0:.0f}  fs={fs:g}')

t0_eda = read_e4_header(os.path.join(sdir, 'EDA.csv'))[0]
t0_hr  = read_e4_header(os.path.join(sdir, 'HR.csv'))[0]
print(f'\nHR starts {t0_hr - t0_eda:.0f} s after EDA  <-- never assume a shared t0')

eda = read_signal(os.path.join(sdir, 'EDA.csv'), 'EDA')
acc = read_signal(os.path.join(sdir, 'ACC.csv'), 'ACC')
ibi = read_ibi(os.path.join(sdir, 'IBI.csv'))
print(f'EDA {len(eda):,} @4Hz = {len(eda)/4/3600:.3f} h   ACC {acc.shape}   IBI {len(ibi)} beats')
print('NaNs in EDA:', int(eda.isna().sum()), '<-- the E4 pads nothing; gaps are *missing files*, not NaNs')

## Step 2 — Session index and the fragmentation problem

The folder-name epoch **equals** the in-file `t0` (verified across the archive), so a full
index can be built from names + file sizes without opening 609 × 6 CSVs. `exact=True`
counts lines instead — slower, but authoritative. Use the estimate to explore and the exact
pass once before you commit to a processing run.

**The headline result:** median session is 1.17 h and 78 sessions are under 3 minutes.
A nurse's shift is scattered across many folders. Any pipeline built on "one file per
subject" or "one 8-hour recording" is modelling a dataset that does not exist.

In [ ]:
EDA_BYTES_PER_SAMPLE, EDA_HEADER_BYTES = 9.0, 27.0   # "0.123456\n"; exact unless EDA >= 10 uS

def build_session_index(root, exact=False):
    rows = []
    for eda_path in sorted(glob.glob(os.path.join(root, '*', '*', 'EDA.csv'))):
        sd = os.path.dirname(eda_path)
        folder, subject = os.path.basename(sd), os.path.basename(os.path.dirname(sd))
        m = re.match(r'^(?P<sub>.+)_(?P<ep>\d{9,11})$', folder)
        ep_name = int(m.group('ep')) if m else np.nan
        t0, fs = read_e4_header(eda_path)
        if exact:
            secs = (sum(1 for _ in open(eda_path)) - 2) / fs
        else:
            secs = ((os.path.getsize(eda_path) - EDA_HEADER_BYTES) / EDA_BYTES_PER_SAMPLE) / fs
        rows.append(dict(subject=subject, folder=folder, path=sd,
                         epoch_name=ep_name, epoch_file=t0, seconds=secs,
                         **{f'{n}_bytes': os.path.getsize(os.path.join(sd, n + '.csv'))
                            for n in ['EDA', 'HR', 'IBI', 'ACC', 'BVP', 'TEMP', 'tags']}))
    S = pd.DataFrame(rows)
    S['start'] = pd.to_datetime(S.epoch_file, unit='s', utc=True)
    S['end']   = S.start + pd.to_timedelta(S.seconds, unit='s')
    S['start_local'] = S.start.dt.tz_convert(LOCAL_TZ)
    return S.sort_values(['subject', 'start']).reset_index(drop=True)

S = build_session_index(ROOT)
print(f'{len(S)} sessions | {S.seconds.sum()/3600:,.0f} sensor hours')
print('folder epoch == in-file epoch for all sessions:', bool((S.epoch_name == S.epoch_file).all()))
print('tags.csv empty: %d/%d (%.0f%%)' % ((S.tags_bytes == 0).sum(), len(S), 100*(S.tags_bytes == 0).mean()))

In [ ]:
h = S.seconds / 3600
print('session duration (h): median %.2f  mean %.2f  p05 %.3f  p95 %.2f  max %.2f'
      % (h.median(), h.mean(), h.quantile(.05), h.quantile(.95), h.max()))
print('\n', pd.cut(h, [0, .05, .25, 1, 2, 4, 8, 13]).value_counts().sort_index().to_string())

# gaps: 9% of consecutive sessions are <60 s apart -> the device was stopped and restarted
S['gap_s'] = S.groupby('subject').apply(
    lambda g: (g.start.shift(-1) - g.end).dt.total_seconds(), include_groups=False).values
g = S.gap_s.dropna()
print('\ngap to next session: median %.0f s | <60s %.0f%% | <5min %.0f%% | NEGATIVE (overlap) %d'
      % (g.median(), 100*(g < 60).mean(), 100*(g < 300).mean(), (g < 0).sum()))

fig, ax = plt.subplots(1, 2, figsize=(14, 3.2))
h.plot.hist(bins=60, ax=ax[0], title='Session duration (h)')
S.start_local.dt.hour.value_counts().sort_index().plot.bar(
    ax=ax[1], title='Session start hour (local) — day/night shift mix'); plt.tight_layout()

**Act on this:** 7 pairs of sessions *overlap in time for the same subject* — two recordings
claiming the same wall clock. Deduplicate before joining, or a labelled second can be counted
twice. Sessions separated by <60 s should be stitched; sessions under a few minutes carry
almost no usable window and are usually best dropped.

In [ ]:
S['drop_tiny']  = S.seconds < 300
S['overlap_prev'] = S.groupby('subject').apply(
    lambda x: x.start < x.end.shift(), include_groups=False).values
print('flagged tiny: %d   overlapping: %d' % (S.drop_tiny.sum(), S.overlap_prev.sum()))
display(S[S.overlap_prev][['subject','folder','start_local','seconds']])

## Step 3 — Coverage census: the 15 subjects are not 15 equal units

Uneven by a factor of six in hours, and by a factor of ten in usable IBI. `IBI.csv` size is
the single best predictor of whether HRV features are possible for a subject.

In [ ]:
cen = S.groupby('subject').agg(
    sessions=('seconds', 'size'), hours=('seconds', lambda x: x.sum()/3600),
    tiny=('drop_tiny', 'sum'), empty_tags=('tags_bytes', lambda x: (x == 0).sum()),
    ibi_MB=('IBI_bytes', lambda x: x.sum()/1e6), eda_MB=('EDA_bytes', lambda x: x.sum()/1e6))
cen['ibi_per_hour_kB'] = cen.ibi_MB*1e3 / cen.hours     # HRV feasibility proxy
print(cen.round(2).sort_values('hours').to_string())
cen.hours.sort_values().plot.barh(title='Sensor hours per subject', figsize=(7, 4));

## Step 4 — Timezone: settle it with evidence, not documentation

Sensors are UTC epochs; survey times are bare local wall-clock. The offset is testable:
localise the survey with a candidate offset and count how many events land **fully inside**
one of that subject's own sensor sessions. The true offset should stand out sharply.

Measured on the full archive:

| assumed local offset | events fully inside a session |
|---|---|
| UTC−0 | 58 / 358 |
| UTC−4 | 153 / 358 |
| **UTC−5** | **283 / 358 (79%)** |
| UTC−6 | 198 / 358 |
| UTC−7 | 154 / 358 |

UTC−5 wins decisively → US **Central** time (the hospital is in Louisiana). Then, DST-aware
`America/Chicago` beats a fixed −5 offset: **305 vs 283**. Because the data spans April to
December, a fixed offset misaligns the ~22 winter events by a full hour. Use the named zone.

In [ ]:
FACTORS = ['COVID related', 'Treating a covid patient', 'Patient in Crisis',
           "Patient or patient's family", 'Doctors or colleagues',
           'Administration, lab, pharmacy, radiology, or other ancilliary services',
           'Increased Workload', 'Technology related stress', 'Lack of supplies',
           'Documentation', 'Competency related stress',
           'Saftey (physical or physiological threats)',
           'Work Environment - Physical or others: work processes or procedures']

def load_survey(path, tz=LOCAL_TZ):
    df = pd.read_excel(path, sheet_name=0)
    df.columns = [re.sub(r'\s+', ' ', str(c)).strip() for c in df.columns]  # kills the trailing \n
    df['subject'] = df['ID'].astype(str).str.strip()                        # int 15 -> '15'
    mk = lambda d, t: pd.Timestamp(dt.datetime.combine(pd.Timestamp(d).date(), t))
    df['start_local'] = [mk(d, t) for d, t in zip(df['date'], df['Start time'])]
    df['end_local']   = [mk(d, t) for d, t in zip(df['date'], df['End time'])]
    df['crossed_midnight'] = df.end_local < df.start_local
    df.loc[df.crossed_midnight, 'end_local'] += pd.Timedelta(days=1)
    for c in ['start_local', 'end_local']:
        df[c] = df[c].dt.tz_localize(tz, nonexistent='shift_forward', ambiguous=True)
        df[c + '_utc'] = df[c].dt.tz_convert('UTC')
    for c in ['Stress level'] + [re.sub(r'\s+', ' ', f).strip() for f in FACTORS]:
        if c in df:                                   # 'na' string -> real NaN
            df[c] = pd.to_numeric(df[c].replace({'na': np.nan, 'NA': np.nan}), errors='coerce')
    df['labelled'] = df['Stress level'].notna()
    df['duration_min'] = (df.end_local_utc - df.start_local_utc).dt.total_seconds() / 60
    df['is_dup'] = df.duplicated(['subject', 'date', 'Start time', 'End time'], keep=False)
    return df

SV = load_survey(SURVEY)
print(SV.shape, '| labelled:', int(SV.labelled.sum()), '| duplicate rows:', int(SV.is_dup.sum()))
print('subjects in survey but not on disk:', set(SV.subject) - set(S.subject))

In [ ]:
def offset_sweep(sv, sess, offsets=range(0, 9)):
    win = {k: v[['start', 'end']].values for k, v in sess.groupby('subject')}
    out = []
    for off in offsets:
        hit = 0
        for sub, s, e in zip(sv.subject, sv.start_local.dt.tz_localize(None),
                                          sv.end_local.dt.tz_localize(None)):
            w = win.get(sub)
            if w is None: continue
            su = pd.Timestamp(s, tz='UTC') + pd.Timedelta(hours=off)
            eu = pd.Timestamp(e, tz='UTC') + pd.Timedelta(hours=off)
            hit += bool(((w[:, 0] <= su) & (w[:, 1] >= eu)).any())
        out.append((f'UTC-{off}', hit, round(100*hit/len(sv), 1)))
    return pd.DataFrame(out, columns=['assumed_local', 'events_fully_inside', 'pct'])

print(offset_sweep(SV, S).to_string(index=False))

## Step 5 — Label reality check

This is where the analysis plan is actually decided.

- **358 events total; 245 labelled.** That is the effective sample size — not the 18 million
  4 Hz samples. Per subject: 4 (`6D`) to 46 (`7A`).
- **`Stress level` is 0/1/2 with counts 46 / 20 / 179.** There is no "not stressed" row
  anywhere in the file. Every row is an event the nurse flagged; the level is its severity.
  So the *majority* labelled class is the most severe one, and the middle class has 20 examples
  across the whole study. **You must construct the negative class yourself** from unlabelled time.
- **9 of 15 subjects have zero level-1 events.** Three-class LOSO is close to unworkable:
  a held-out subject will often contain classes the model never saw, or lack classes it predicts.
- **Only 12.3% of sensor time falls inside any event, and 64.5% of sessions contain none.**
- **Event durations:** median 18 min, mean 29 min, **max 323 min**, 34 events over an hour.
  This is the concrete form of the published criticism that marked sections exceed plausible
  event length. 12 event pairs overlap within a subject; 3 rows are exact duplicates.
- **`COVID related` is checked exactly once in 245 labelled events** — treat as broken and drop.
  Mean factors per event is 1.02 and 79 events have none, so this is effectively single-label.

In [ ]:
print('stress level:', SV['Stress level'].value_counts(dropna=False).to_dict())
per = SV.pivot_table(index='subject', columns='Stress level', aggfunc='size', fill_value=0)
per['events'] = SV.groupby('subject').size(); per['labelled'] = SV.groupby('subject').labelled.sum()
print('\n', per.to_string())
print('\nsubjects with ZERO level-1 events:', (per.get(1.0, pd.Series(0, per.index)) == 0).sum(), 'of 15')

d = SV.duration_min
print('\nduration (min): med %.0f mean %.0f p90 %.0f max %.0f | >60min: %d | <=2min: %d'
      % (d.median(), d.mean(), d.quantile(.9), d.max(), (d > 60).sum(), (d <= 2).sum()))
fig, ax = plt.subplots(1, 2, figsize=(14, 3.2))
d.clip(upper=120).plot.hist(bins=60, ax=ax[0], title='Event duration (min, clipped at 120)')
SV[SV.labelled][[re.sub(r'\s+',' ',f).strip() for f in FACTORS]].sum().sort_values().plot.barh(
    ax=ax[1], title='Factor prevalence (labelled events)'); plt.tight_layout()

In [ ]:
# overlapping and duplicated events -- resolve before any join
ov = []
for sub, g in SV.sort_values('start_local').groupby('subject'):
    g = g.reset_index()
    for i in range(len(g) - 1):
        if g.loc[i+1, 'start_local'] < g.loc[i, 'end_local']:
            ov.append((sub, g.loc[i, 'start_local'], g.loc[i, 'end_local'],
                       g.loc[i+1, 'start_local'], g.loc[i+1, 'end_local']))
print('overlapping event pairs:', len(ov))
display(pd.DataFrame(ov, columns=['subject','a_start','a_end','b_start','b_end']).head(12))

### How much sensor time is actually labelled, per subject

In [ ]:
rows = []
for _, r in S.iterrows():
    e = SV[(SV.subject == r.subject) & (SV.end_local_utc > r.start) & (SV.start_local_utc < r.end)]
    sec = sum(max(0, (min(x.end_local_utc, r.end) - max(x.start_local_utc, r.start)).total_seconds())
              for _, x in e.iterrows())
    rows.append((r.subject, r.seconds, len(e), sec))
COV = pd.DataFrame(rows, columns=['subject', 'sensor_s', 'n_events', 'event_s'])
t = COV.groupby('subject').agg(hours=('sensor_s', lambda x: x.sum()/3600),
                               event_hours=('event_s', lambda x: x.sum()/3600),
                               sess_with_event=('n_events', lambda x: (x > 0).sum()),
                               sessions=('n_events', 'size'))
t['pct_labelled'] = 100 * t.event_hours / t.hours
print(t.round(2).to_string())
print('\nOVERALL: %.1f%% of sensor time is inside an event; %d/%d sessions have none'
      % (100*COV.event_s.sum()/COV.sensor_s.sum(), (COV.n_events == 0).sum(), len(COV)))

## Step 6 — Resample to 1 Hz with **per-signal** aggregators

A blanket `.resample('1s').mean()` destroys two of the six channels:

- **ACC** — the signed mean is dominated by gravity. Measured on one session, the per-axis
  means are `[0.481, 0.055, 0.465] g` (orientation), while mean magnitude is `1.005 g` and the
  within-second SD is `0.096 g`. The motion energy you need as an exertion covariate lives in
  the **variance**, which averaging deletes. Use magnitude mean, SD, and peak-to-peak.
- **BVP** — a zero-centred waveform: measured mean `0.0001`, SD `23.55`. Its 1 Hz mean is noise.
  Use RMS, or leave BVP out and take HR/IBI as its summaries.

Also note `pandas >= 2.2` deprecates `'1S'`; the lowercase `'1s'` is correct.

In [ ]:
def session_to_1hz(sdir, with_bvp_rms=False):
    """Per-signal aggregation onto a shared 1 Hz UTC grid. Each signal carries its own t0,
    so the union index absorbs HR's 10 s offset automatically."""
    out = {}
    for n in ['EDA', 'TEMP', 'HR']:
        p = os.path.join(sdir, n + '.csv')
        if os.path.exists(p) and os.path.getsize(p) > 30:
            out[n.lower()] = read_signal(p, n).resample('1s').mean()
    p = os.path.join(sdir, 'ACC.csv')
    if os.path.exists(p) and os.path.getsize(p) > 60:
        mag = np.sqrt((read_signal(p, 'ACC') ** 2).sum(axis=1))
        r = mag.resample('1s')
        out['acc_mag'], out['acc_sd'], out['acc_p2p'] = r.mean(), r.std(), r.max() - r.min()
    df = pd.DataFrame(out)
    if with_bvp_rms:
        p = os.path.join(sdir, 'BVP.csv')
        if os.path.exists(p) and os.path.getsize(p) > 60:
            b = read_signal(p, 'BVP')
            df['bvp_rms'] = ((b ** 2).resample('1s').mean() ** .5).reindex(df.index)
    p = os.path.join(sdir, 'IBI.csv')
    if os.path.exists(p) and os.path.getsize(p) > 20:
        ibi = read_ibi(p)
        if len(ibi):
            df['ibi'] = ibi.resample('1s').mean().reindex(df.index)
            df['ibi_present'] = (ibi.resample('1s').count().reindex(df.index).fillna(0) > 0)
    return df

D = session_to_1hz(sdir)
print(D.shape); print(D.describe().T[['mean', '50%', 'min', 'max']].round(3).to_string())

## Step 7 — Non-wear detection, and a heuristic that **failed**

The textbook rule is "skin temperature below ~30 °C means the band is off the wrist."
**On this dataset that rule flags 100% of the session.** Measured on `5C_1587297777`:
TEMP runs 23.5–29.1 °C for the entire 8.9 h, with 80% of seconds below 28 °C, while EDA is
below 0.05 µS for only **0.6%** of the time. The E4's thermopile reads well below true skin
temperature under an air-conditioned hospital band, so an absolute threshold is meaningless here.

Use instead: EDA at the floor **and** accelerometer stillness, with TEMP contributing only as a
*relative* drop against its own rolling median. Require a minimum run length so single-second
dropouts aren't flagged.

Do **not** rely on `.isnull()` for missingness — the E4 pads nothing. Measured NaN count across
EDA/TEMP/HR/BVP for a full session: **zero**. A missingness heatmap here returns a clean bill of
health that is false.

In [ ]:
def flag_nonwear(df, eda_floor=0.05, still_sd=0.005, temp_drop=1.5, min_run_s=60):
    eda_off = df['eda'] < eda_floor if 'eda' in df else pd.Series(False, df.index)
    still   = df['acc_sd'] < still_sd if 'acc_sd' in df else pd.Series(False, df.index)
    tdev    = pd.Series(False, df.index)
    if 'temp' in df:                     # RELATIVE, not absolute -- see markdown above
        tdev = (df['temp'].rolling('30min', min_periods=60).median() - df['temp']) > temp_drop
    raw = eda_off & (still | tdev)
    grp = (raw != raw.shift()).cumsum()  # suppress runs shorter than min_run_s
    return (raw & raw.groupby(grp).transform('size').ge(min_run_s)).rename('nonwear')

nw = flag_nonwear(D)
print('non-wear: %d s (%.2f%%)' % (nw.sum(), 100 * nw.mean()))
raw_nans = sum(int(read_signal(os.path.join(sdir, n + '.csv'), n).isna().sum())
               for n in ['EDA', 'TEMP', 'HR', 'BVP'])
print('naive TEMP<30C would flag: %.1f%%   |   NaNs in the RAW files: %d' % (100 * (D.temp < 30).mean(), raw_nans))
print('NaNs in the joined 1 Hz frame: %d  <-- introduced by HRs 10 s offset, not by the sensors'
      % int(D[['eda', 'temp', 'hr']].isna().to_numpy().sum()))
print('\nEDA on worn seconds: med %.3f  p99 %.3f  skew %.2f'
      % (D.eda[~nw].median(), D.eda[~nw].quantile(.99), D.eda[~nw].skew()))

**A second prediction that failed:** EDA is often described as heavily right-skewed, motivating a
log transform. Measured here it is **near-symmetric** (skew −0.07, median 1.41 µS, range
0.05–3.73). Check skew per subject before transforming; don't apply a log because the
literature says so.

## Step 8 — IBI is far sparser than the file's existence suggests

Measured on `5C_1587297777`: **1,050 beats over 8.94 h = 2.8% of the ~37,500 expected** at
70 bpm. Median gap between retained beats is **6.4 s** (consecutive beats would be ~0.9 s), and
there are **zero** contiguous runs of ≥30 beats with gaps ≤2 s.

**HRV features are not computable for this session at all**, and `IBI.csv` exists in all 609
sessions regardless. Run this check per subject before planning any HRV work; the census in
Step 3 (`ibi_per_hour_kB`) tells you where to bother looking.

In [ ]:
def ibi_usability(path, min_run=30, max_gap=2.0):
    ibi = read_ibi(path)
    if len(ibi) < 2:
        return dict(beats=len(ibi), usable_runs=0, longest_run=0, median_gap=np.nan)
    gaps = np.diff(ibi.index.view('int64') / 1e9)
    runs = [len(r) for r in np.split(np.arange(len(gaps)), np.where(gaps > max_gap)[0] + 1)]
    return dict(beats=len(ibi), usable_runs=sum(r >= min_run for r in runs),
                longest_run=max(runs) if runs else 0, median_gap=float(np.median(gaps)))

print(ibi_usability(os.path.join(sdir, 'IBI.csv')))
samp = S.sort_values('IBI_bytes', ascending=False).groupby('subject').head(1)
rep = pd.DataFrame([{'subject': r.subject, 'hours': round(r.seconds/3600, 2),
                     **ibi_usability(os.path.join(r.path, 'IBI.csv'))}
                    for _, r in samp.iterrows()]).set_index('subject')
print('\nbest-IBI session per subject:\n', rep.sort_values('usable_runs', ascending=False).to_string())

## Step 9 — Interval label join with an explicit third state

`ffill` on a label column would propagate "stressed" across the hours between events. The join
must be an interval match, and unlabelled time must stay **unlabelled** — a third state, not a
zero. A guard band around each event is excluded from any candidate-baseline pool, because the
published criticism of this dataset is precisely that some events have no cool-down separating
them from baseline.

In [ ]:
def attach_labels(df, subject, survey, guard_min=10, drop_dups=True):
    ev = survey[survey.subject == subject]
    if drop_dups:
        ev = ev.drop_duplicates(['subject', 'date', 'Start time', 'End time'])
    state = pd.Series('unlabelled', index=df.index, dtype=object)
    lvl   = pd.Series(np.nan, index=df.index)
    eid   = pd.Series(np.nan, index=df.index)
    guard = pd.Series(False, index=df.index)
    for i, r in ev.iterrows():
        m = (df.index >= r.start_local_utc) & (df.index <= r.end_local_utc)
        state[m], lvl[m], eid[m] = 'event', r['Stress level'], i
        guard |= ((df.index >= r.start_local_utc - pd.Timedelta(minutes=guard_min)) &
                  (df.index <= r.end_local_utc   + pd.Timedelta(minutes=guard_min)))
    state[(state == 'unlabelled') & guard] = 'guard'
    out = df.copy()
    out['label_state'], out['stress_level'], out['event_id'] = state, lvl, eid
    out['unlabelled_level'] = out.event_id.notna() & out.stress_level.isna()   # the 113 'na' events
    return out

subject = os.path.basename(os.path.dirname(sdir))
DL = attach_labels(D, subject, SV)
print(DL.label_state.value_counts().to_dict())
print('candidate baseline seconds (unlabelled & worn):', int(((DL.label_state == "unlabelled") & ~nw).sum()))

## Step 10 — Look at individual events before engineering a single feature

Ten minutes here catches misalignment, exertion-driven events, and implausible durations faster
than any summary statistic. Watch for EDA rising a few seconds *after* onset — skin conductance
responses lag by roughly 1–4 s, so a window starting exactly at the marked onset sits mostly in
pre-response signal. And watch `acc_mag`: if it climbs with HR, you may be looking at physical
exertion rather than psychological stress.

In [ ]:
def plot_event(DL, event_id, pad_min=20):
    r = SV.loc[event_id]
    w = DL.loc[r.start_local_utc - pd.Timedelta(minutes=pad_min):
               r.end_local_utc   + pd.Timedelta(minutes=pad_min)]
    if w.empty:
        print('no sensor coverage for event', event_id); return
    cols = [c for c in ['eda', 'hr', 'temp', 'acc_mag'] if c in w]
    fig, ax = plt.subplots(len(cols), 1, figsize=(14, 2.1*len(cols)), sharex=True)
    for a, c in zip(np.atleast_1d(ax), cols):
        a.plot(w.index, w[c], lw=.8); a.set_ylabel(c)
        a.axvspan(r.start_local_utc, r.end_local_utc, color='orange', alpha=.25)
    np.atleast_1d(ax)[0].set_title(
        f'{r.subject} | event {event_id} | level={r["Stress level"]} | {r.duration_min:.0f} min')
    plt.tight_layout()

ids = SV[(SV.subject == subject)].index
for e in ids[:3]:
    plot_event(DL, e)

## Step 11 — Per-subject distributions and variance decomposition

Pooled histograms mostly encode subject identity. ICC quantifies exactly how much, and thereby
decides whether per-subject normalisation is optional or mandatory. Normalise **causally** —
a full-recording z-score uses the events themselves as the reference distribution and leaks
test-fold statistics into training.

In [ ]:
def causal_z(s, window='60min', min_periods=600):
    """Trailing robust z. Uses only the past -> no leakage, no contamination by the event."""
    med = s.rolling(window, min_periods=min_periods).median()
    iqr = (s.rolling(window, min_periods=min_periods).quantile(.75) -
           s.rolling(window, min_periods=min_periods).quantile(.25))
    return (s - med) / iqr.replace(0, np.nan)

def icc1(long, value, group='subject'):
    g = long.groupby(group)[value]; k = g.count(); m = g.mean()
    if len(k) < 2 or len(long) <= len(k):
        return np.nan                      # needs >=2 subjects with >1 obs each
    ms_b = (k * (m - long[value].mean()) ** 2).sum() / (len(k) - 1)
    ms_w = g.apply(lambda x: ((x - x.mean()) ** 2).sum()).sum() / (len(long) - len(k))
    if not np.isfinite(ms_w) or ms_w <= 0:
        return np.nan
    var_b = max((ms_b - ms_w) / k.mean(), 0)
    return var_b / (var_b + ms_w)

for c in ['eda', 'hr', 'temp']:
    if c in DL: DL[c + '_z'] = causal_z(DL[c])
print(DL[[c for c in DL if c.endswith("_z")]].describe().T[['mean','std','min','max']].round(2).to_string())

In [ ]:
# Pool a sample of sessions per subject, then measure how much variance is 'who' vs 'when'.
# Raise N_PER_SUBJECT once you're happy with runtime.
N_PER_SUBJECT = 3
parts = []
for sub, g in S[~S.drop_tiny].sort_values('seconds', ascending=False).groupby('subject'):
    for _, r in g.head(N_PER_SUBJECT).iterrows():
        try:
            d = session_to_1hz(r.path)
            d = d[~flag_nonwear(d)].resample('60s').mean()
            d['subject'] = sub; parts.append(d)
        except Exception as ex:
            print('skip', r.folder, ex)
L = pd.concat(parts).dropna(subset=['eda'])
print(L.shape)
for c in ['eda', 'hr', 'temp', 'acc_mag']:
    if c in L: print(f'  ICC(subject) for {c:8} = {icc1(L.dropna(subset=[c]), c):.3f}')

fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
for a, c in zip(ax, ['eda', 'hr', 'temp']):
    L.boxplot(column=c, by='subject', ax=a, grid=False); a.set_title(c); a.set_xlabel('')
plt.suptitle('Per-subject distributions — pooling these would be misleading'); plt.tight_layout()

## Step 12 — Split design and effective sample size

The numbers that should govern every modelling decision downstream:

| | |
|---|---|
| 1 Hz samples (worn) | ~4.5 million |
| Labelled events | **245** |
| Events at level 1 | **20** |
| Subjects | **15** (9 with no level-1 event at all) |

Sample count is not sample size. Two rules follow:

- **Group by subject.** `GroupKFold` / `LeaveOneGroupOut` on `subject`. A random split over
  windows leaks near-duplicate neighbours across the boundary and inflates every metric.
- **Do not SMOTE.** Interpolating between adjacent, autocorrelated 1 Hz windows manufactures
  copies of real samples; applied before the split (the usual mistake) it leaks outright. With
  20 examples in the middle class, it inflates scores rather than improving the model. Use class
  weights, threshold tuning, and event-level metrics.

Given the class structure, binary **level 2 vs. constructed baseline** is far more defensible than
3-class. Score at the **event** level — an event counts as detected if enough of its windows fire —
and report the number of events in each fold, because a fold with 3 positive events cannot support
a confident F1.

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold   # noqa

ev = SV[SV.labelled].copy()
ev['pos'] = (ev['Stress level'] == 2).astype(int)
print('binary task: %d positive (level 2) vs %d other labelled events' % (ev.pos.sum(), (~ev.pos.astype(bool)).sum()))
logo = LeaveOneGroupOut()
print('\nfold-by-fold event counts (held-out subject):')
for tr, te in logo.split(ev, ev.pos, groups=ev.subject):
    h = ev.iloc[te]
    print(f'  {h.subject.iloc[0]:>3}: test n={len(h):3d} pos={h.pos.sum():3d} | train n={len(tr):3d} pos={ev.iloc[tr].pos.sum():3d}')

### Caching, and what to do next

Reparsing 3.5 GB per iteration is the main thing that will slow you down. Cache the 1 Hz frames
to Parquet once, keyed by session folder, and everything after Step 6 becomes fast.

Remaining work this notebook deliberately leaves open:

1. **EDA tonic/phasic decomposition** (`neurokit2.eda_phasic`, or cvxEDA). SCR frequency and
   amplitude are the physiologically meaningful quantities; raw conductance is not. Worth doing
   once the label join is trusted, not before.
2. **A documentation discrepancy to resolve with the authors.** The paper describes two
   collection windows (Apr–May and Nov–Dec 2020), but both the session folders and the survey
   dates run continuously from 2020-04-14 to 2020-12-13 — 239 of 358 events and 288 of 609
   sessions fall in Jun–Aug, outside the documented windows. Sensors and survey agree, so this
   is not a parsing artifact. Establish whether the extra months are a third undocumented
   session before publishing anything that cites the collection period.
3. **The 46 sessions that do have button presses.** `tags.csv` is empty in 563 of 609, and two
   subjects (`83`, `F5`) have none at all — but where tags exist they are an independent second
   source of event onsets and a direct test of the timezone conclusion.

In [ ]:
def cache_session(r, force=False):
    fp = os.path.join(CACHE, f'{r.subject}__{r.folder}.parquet')
    if os.path.exists(fp) and not force:
        return fp
    d = session_to_1hz(r.path)
    d['nonwear'] = flag_nonwear(d)
    d = attach_labels(d, r.subject, SV)
    try:
        d.to_parquet(fp)                      # needs pyarrow or fastparquet
    except ImportError:
        fp = fp.replace('.parquet', '.pkl')   # graceful fallback
        d.to_pickle(fp)
    return fp

for _, r in S[~S.drop_tiny].head(5).iterrows():
    print('cached', os.path.basename(cache_session(r)))